## Comparative Analysis & Results: Recursive Training vs. Human Control

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

RESULTS_DIR = Path("../../final_qwen2.5-3b").resolve()
# Final results directories
FINAL_RESULTS_DIR = Path.cwd() / "final_results"
DATA_DIR = FINAL_RESULTS_DIR /"data"
PLOTS_DIR = FINAL_RESULTS_DIR/"plots"

# Create directories if they don't already exist
FINAL_RESULTS_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR.mkdir(parents=True, exist_ok=True)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

HUMAN_CONTROL_42 = RESULTS_DIR / "seeds/seed_42/human_control/evaluation"
HUMAN_CONTROL_123 = RESULTS_DIR / "seeds/seed_123/human_control/evaluation"
RECURSIVE_42 = RESULTS_DIR / "seeds/seed_42/recursive/evaluation"
RECURSIVE_123 = RESULTS_DIR / "seeds/seed_123/recursive/evaluation"

# Paths to the 4 evaluation run directories
RUN_PATHS = {
    ("Recursive", 42): RECURSIVE_42,
    ("Recursive", 123): RECURSIVE_123,
    ("Human_Control", 42): HUMAN_CONTROL_42,
    ("Human_Control", 123): HUMAN_CONTROL_123,
}

METRIC_COLUMNS = [
    "disease_f1", "chemical_f1", "disease_retention", "chemical_retention",
    "context_supported_rate", "context_unsupported_rate",
    "bleu_1", "bleu_4", "rouge_l", "meteor", "chrf_pp",
    "bertscore_f1", "cosine_similarity",
    "word_count", "token_count", "repetition_rate", "perplexity"
]

records = []
for (condition, seed), folder in RUN_PATHS.items():
    df = pd.read_csv(folder / "combined_lexical_semantic_domain_per_sample.csv")

    # Impute 0.0 when gold entities exist but zero were matched (avoids survivorship bias)
    for g in ["g0", "g1", "g2", "g3"]:
        if "gold_disease_count" in df.columns:
            df.loc[df["gold_disease_count"] > 0, f"{g}_disease_f1"] = (
                df.loc[df["gold_disease_count"] > 0, f"{g}_disease_f1"].fillna(0.0)
            )
        if "gold_chemical_count" in df.columns:
            df.loc[df["gold_chemical_count"] > 0, f"{g}_chemical_f1"] = (
                df.loc[df["gold_chemical_count"] > 0, f"{g}_chemical_f1"].fillna(0.0)
            )

    for g_num in range(4):
        gen = f"G{g_num}"
        prefix = f"g{g_num}"
        row = {"condition": condition, "seed": seed, "generation": gen}
        for m in METRIC_COLUMNS:
            col_name = f"{prefix}_{m}"
            row[m] = df[col_name].mean() if col_name in df.columns else np.nan
        records.append(row)

master_df = pd.DataFrame(records)
display(master_df.head(5))
output_file = DATA_DIR / "cross_condition_cross_seed_master.csv"
master_df.to_csv(output_file, index=False)

In [ ]:
%pip install seaborn

## Comparing Multiple seeds

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

# Load the master summary table you verified
df = pd.read_csv(DATA_DIR / "cross_condition_cross_seed_master.csv")

## Cross condtion and multi seed for disease f1, chemical f1, disease retention, chemical retention, bleu 1, rouge l

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns


def plot_all_seeds_cross_trajectory(
    csv_path=DATA_DIR / "cross_condition_cross_seed_master.csv",
    output_prefix= PLOTS_DIR / "all_seeds_cross_trajectory",
):
  df = pd.read_csv(csv_path)

  df["run_label"] = (
      df["condition"].str.replace("_", " ")
      + " (Seed "
      + df["seed"].astype(str)
      + ")"
  )

  sns.set_theme(style="whitegrid", font_scale=1.05)
  plt.rcParams["font.sans-serif"] = "DejaVu Sans"

  fig, axes = plt.subplots(2, 3, figsize=(16, 10), sharex=True)

  panels = [
      ("disease_f1", "Disease F1", (0.08, 0.30)),
      ("chemical_f1", "Chemical F1", (0, 0.5)),
      ("disease_retention", "Disease Retention Rate", (0.10, 0.30)),
      (
          "chemical_retention",
          "Chemical Retention",
          (0.0, 0.50),
      ),
      ("bleu_1", "BLEU-1", (0, 0.20)),
      ("rouge_l", "ROUGE-L", (0, 0.20))
  ]

  rec_seeds = sorted(df[df["condition"] == "Recursive"]["seed"].unique())
  ctrl_seeds = sorted(df[df["condition"] == "Human_Control"]["seed"].unique())

  rec_colors = ["#D9534F", "#B52B27", "#8C1B17", "#5C0F0C"][: len(rec_seeds)]
  ctrl_colors = ["#0275D8", "#014C8C", "#01325C", "#001D36"][: len(ctrl_seeds)]

  palette = {}
  markers = {}
  dashes = {}

  for i, s in enumerate(rec_seeds):
    key = f"Recursive (Seed {s})"
    palette[key] = rec_colors[i]
    markers[key] = "o" if i == 0 else "X"
    dashes[key] = "solid" if i == 0 else "dashed"

  for i, s in enumerate(ctrl_seeds):
    key = f"Human Control (Seed {s})"
    palette[key] = ctrl_colors[i]
    markers[key] = "s" if i == 0 else "P"
    dashes[key] = "solid" if i == 0 else "dashed"

  for ax, (metric, title, ylim) in zip(axes.flatten(), panels):
    for run_name, grp in df.groupby("run_label"):
      ax.plot(
          grp["generation"],
          grp[metric],
          label=run_name,
          color=palette[run_name],
          marker=markers[run_name],
          markersize=8,
          linewidth=2.2,
          linestyle=dashes[run_name],
      )
    ax.set_title(title, fontsize=12, fontweight="bold", pad=8)
    ax.set_ylabel("Score / Value", fontsize=10)
    ax.set_xlabel("Generation", fontsize=10)
    ax.set_ylim(ylim)
    ax.legend(
        title="Condition & Seed",
        frameon=True,
        loc="best",
        fontsize=8.5,
        title_fontsize=9,
    )

  plt.suptitle(
      "Cross-Condition & Multi-Seed Generational Trajectories (G0 → G3)\n"
      "[Evaluating Multiple Seeds Simultaneously for Recursive Self-Training"
      " vs. Human Control]",
      fontsize=14,
      fontweight="bold",
      y=0.995,
  )
  plt.tight_layout()
  plt.savefig(f"{output_prefix}_1.png", dpi=300)
  plt.close()

plot_all_seeds_cross_trajectory()
print("Executed successfully.")

## Cross condtion and multi seed for meteor, context_supported_rate, context_unsupported_rate, bertscore_f1, cosine_similarity, bleu_1

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns


def plot_all_seeds_cross_trajectory(
    csv_path=DATA_DIR / "cross_condition_cross_seed_master.csv",
    output_prefix=PLOTS_DIR / "all_seeds_cross_trajectory",
):
  df = pd.read_csv(csv_path)

  df["run_label"] = (
      df["condition"].str.replace("_", " ")
      + " (Seed "
      + df["seed"].astype(str)
      + ")"
  )

  sns.set_theme(style="whitegrid", font_scale=1.05)
  plt.rcParams["font.sans-serif"] = "DejaVu Sans"

  fig, axes = plt.subplots(2, 3, figsize=(16, 10), sharex=True)

  panels = [
      ("meteor", "Meteor", (0.08, 0.20)),
      ("context_supported_rate", "Context Supported Rate", (0, 1)),
      ("context_unsupported_rate", "Context Unsupported Rate", (0, 1)),
      (
          "bertscore_f1",
          "Bert Score F1",
          (0.0, 1),
      ),
      ("bleu_1", "BLEU-1", (0, 0.20)),
      ("cosine_similarity", "Cosine Similarity", (0, 1))
  ]

  rec_seeds = sorted(df[df["condition"] == "Recursive"]["seed"].unique())
  ctrl_seeds = sorted(df[df["condition"] == "Human_Control"]["seed"].unique())

  rec_colors = ["#D9534F", "#B52B27", "#8C1B17", "#5C0F0C"][: len(rec_seeds)]
  ctrl_colors = ["#0275D8", "#014C8C", "#01325C", "#001D36"][: len(ctrl_seeds)]

  palette = {}
  markers = {}
  dashes = {}

  for i, s in enumerate(rec_seeds):
    key = f"Recursive (Seed {s})"
    palette[key] = rec_colors[i]
    markers[key] = "o" if i == 0 else "X"
    dashes[key] = "solid" if i == 0 else "dashed"

  for i, s in enumerate(ctrl_seeds):
    key = f"Human Control (Seed {s})"
    palette[key] = ctrl_colors[i]
    markers[key] = "s" if i == 0 else "P"
    dashes[key] = "solid" if i == 0 else "dashed"

  for ax, (metric, title, ylim) in zip(axes.flatten(), panels):
    for run_name, grp in df.groupby("run_label"):
      ax.plot(
          grp["generation"],
          grp[metric],
          label=run_name,
          color=palette[run_name],
          marker=markers[run_name],
          markersize=8,
          linewidth=2.2,
          linestyle=dashes[run_name],
      )
    ax.set_title(title, fontsize=12, fontweight="bold", pad=8)
    ax.set_ylabel("Score / Value", fontsize=10)
    ax.set_xlabel("Generation", fontsize=10)
    ax.set_ylim(ylim)
    ax.legend(
        title="Condition & Seed",
        frameon=True,
        loc="best",
        fontsize=8.5,
        title_fontsize=9,
    )

  plt.suptitle(
      "Cross-Condition & Multi-Seed Generational Trajectories (G0 → G3)\n"
      "[Evaluating Multiple Seeds Simultaneously for Recursive Self-Training"
      " vs. Human Control]",
      fontsize=14,
      fontweight="bold",
      y=0.995,
  )
  plt.tight_layout()
  plt.savefig(f"{output_prefix}_2.png", dpi=300)
  plt.close()

plot_all_seeds_cross_trajectory()
print("Executed successfully.")

## Cross condtion and multi seed for chrf_pp, word_count, token_count, perplexity

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns


def plot_all_seeds_cross_trajectory(
    csv_path=DATA_DIR / "cross_condition_cross_seed_master.csv",
    output_prefix=PLOTS_DIR / "all_seeds_cross_trajectory",
):
  df = pd.read_csv(csv_path)

  df["run_label"] = (
      df["condition"].str.replace("_", " ")
      + " (Seed "
      + df["seed"].astype(str)
      + ")"
  )

  sns.set_theme(style="whitegrid", font_scale=1.05)
  plt.rcParams["font.sans-serif"] = "DejaVu Sans"

  fig, axes = plt.subplots(2, 3, figsize=(16, 10), sharex=True)

  panels = [
      ("word_count", "Word Count", (30, 100)),
      ("token_count", "Token Count", (30, 100)),
      ("chrf_pp", "chrF++", (20, 30)),
      ("perplexity", "Perplexity", (6, 18)),
      ("bleu_1", "BLEU-1", (0, 0.20)),
      ("cosine_similarity", "Cosine Similarity", (0, 1)),
  ]

  rec_seeds = sorted(df[df["condition"] == "Recursive"]["seed"].unique())
  ctrl_seeds = sorted(df[df["condition"] == "Human_Control"]["seed"].unique())

  rec_colors = ["#D9534F", "#B52B27", "#8C1B17", "#5C0F0C"][: len(rec_seeds)]
  ctrl_colors = ["#0275D8", "#014C8C", "#01325C", "#001D36"][: len(ctrl_seeds)]

  palette = {}
  markers = {}
  dashes = {}

  for i, s in enumerate(rec_seeds):
    key = f"Recursive (Seed {s})"
    palette[key] = rec_colors[i]
    markers[key] = "o" if i == 0 else "X"
    dashes[key] = "solid" if i == 0 else "dashed"

  for i, s in enumerate(ctrl_seeds):
    key = f"Human Control (Seed {s})"
    palette[key] = ctrl_colors[i]
    markers[key] = "s" if i == 0 else "P"
    dashes[key] = "solid" if i == 0 else "dashed"

  for ax, (metric, title, ylim) in zip(axes.flatten(), panels):
    for run_name, grp in df.groupby("run_label"):
      ax.plot(
          grp["generation"],
          grp[metric],
          label=run_name,
          color=palette[run_name],
          marker=markers[run_name],
          markersize=8,
          linewidth=2.2,
          linestyle=dashes[run_name],
      )
    ax.set_title(title, fontsize=12, fontweight="bold", pad=8)
    ax.set_ylabel("Score / Value", fontsize=10)
    ax.set_xlabel("Generation", fontsize=10)
    ax.set_ylim(ylim)
    ax.legend(
        title="Condition & Seed",
        frameon=True,
        loc="best",
        fontsize=8.5,
        title_fontsize=9,
    )

  plt.suptitle(
      "Cross-Condition & Multi-Seed Generational Trajectories (G0 → G3)\n"
      "[Evaluating Multiple Seeds Simultaneously for Recursive Self-Training"
      " vs. Human Control]",
      fontsize=14,
      fontweight="bold",
      y=0.995,
  )
  plt.tight_layout()
  plt.savefig(f"{output_prefix}_3.png", dpi=300)
  plt.close()

plot_all_seeds_cross_trajectory()
print("Executed successfully.")

## Normalized trajectory (index to G0 = 100%)

For every metric, divide each generation's value by that same run's (condition × seed) G0 value, then multiply by 100. This converts everything — whether it's a 0–1 proportion or a 24–97 token count — into "% of starting point," so they're all comparable on one y-axis. A metric at 50 means it's lost half its G0 value; a metric climbing above 100 means it grew relative to G0.

Why it matters here: It answers "which metrics degrade fastest, in relative terms" without you having to bucket metrics by scale first — it sidesteps that whole problem.

One caveat baked into the code: context_unsupported_rate is an "inverse" metric (higher = worse), so I flip its sign convention with a higher_is_better flag — otherwise it'll visually look like it's "improving" when the model is actually getting worse.

At G3, the four highlighted metrics for Recursive land at: disease_f1 45% of G0, context_supported_rate 51%, word_count 143%, perplexity 196% — vs. Human_Control staying near 90–140% on all of them. That's the compressed story: Recursive is losing about half its factual grounding while writing ~1.4x more and getting twice as uncertain, and Human_Control barely moves.

One thing worth flagging from the full sort: repetition_rate and token_count swing wildly (both conditions cross 100%+ by a lot) because their G0 baseline values are tiny/noisy, so % change gets exaggerated — worth treating those two with caution in this particular chart type.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# --- Pick the metrics you care about here ---
METRICS = ["disease_f1", "context_supported_rate", "word_count", "perplexity"]

df = pd.read_csv(DATA_DIR / "cross_condition_cross_seed_master.csv")

# Average the two seeds together so we get one line per condition
df = df.groupby(["condition", "generation"])[METRICS].mean().reset_index()

# Index every metric to its G0 value = 100
g0 = df[df["generation"] == "G0"].set_index("condition")
for metric in METRICS:
  df[metric] = df.apply(
      lambda row: 100 * row[metric] / g0.loc[row["condition"], metric],
      axis=1,
  )

# Plot: one line per condition x metric
fig, ax = plt.subplots(figsize=(9, 6))

colors = {"Recursive": "tab:red", "Human_Control": "tab:blue"}
styles = {m: s for m, s in zip(METRICS, ["-", "--", "-.", ":"])}

for condition, grp in df.groupby("condition"):
  for metric in METRICS:
    ax.plot(
        grp["generation"], grp[metric],
        color=colors[condition],
        linestyle=styles[metric],
        marker="o",
        label=f"{condition} - {metric}",
    )

ax.axhline(100, color="black", linewidth=1, linestyle="--", alpha=0.5)
ax.set_xlabel("Generation")
ax.set_ylabel("% of G0 value")
ax.set_title("Selected Metrics Indexed to G0 = 100%")
ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig(PLOTS_DIR / "normalized_trajectory_simple.png", dpi=300)
plt.close()

print("Saved normalized_trajectory_simple.png")

## Heatmaps

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

# --- Pick the metrics you care about here ---
METRICS = [
    "disease_f1", "chemical_f1", "context_supported_rate",
    "bleu_1", "rouge_l", "word_count", "perplexity",
]

df = pd.read_csv(DATA_DIR/ "cross_condition_cross_seed_master.csv")

# Average the two seeds together so each condition has one value per generation
df = df.groupby(["condition", "generation"])[METRICS].mean().reset_index()

# Index every metric to its own G0 value = 100 (so all metrics share one color scale)
g0 = df[df["generation"] == "G0"].set_index("condition")
for metric in METRICS:
  df[metric] = df.apply(
      lambda row: 100 * row[metric] / g0.loc[row["condition"], metric],
      axis=1,
  )

conditions = sorted(df["condition"].unique())
fig, axes = plt.subplots(1, len(conditions), figsize=(6 * len(conditions), 5), sharey=True)

for ax, condition in zip(axes, conditions):
  sub = df[df["condition"] == condition].set_index("generation")[METRICS].T
  sns.heatmap(
      sub,
      ax=ax,
      annot=True, fmt=".0f",
      cmap="RdBu", center=100, vmin=50, vmax=150,
      cbar_kws={"label": "% of G0"},
  )
  ax.set_title(condition)
  ax.set_xlabel("Generation")
  ax.set_ylabel("Metric" if ax is axes[0] else "")

plt.suptitle("Metric Drift by Generation (% of G0 baseline)", fontweight="bold")
plt.tight_layout()
plt.savefig(PLOTS_DIR/"heatmap_simple.png", dpi=300)
plt.close()

print("Saved heatmap_simple.png")

## Interesting data point

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

df = pd.read_csv(DATA_DIR/"cross_condition_cross_seed_master.csv")

# Average the two seeds together
g = df.groupby(["condition", "generation"])["context_supported_rate"].mean().reset_index()

g0 = g[g["generation"] == "G0"].set_index("condition")["context_supported_rate"]
g3 = g[g["generation"] == "G3"].set_index("condition")["context_supported_rate"]

fig, ax = plt.subplots(figsize=(6, 7))

colors = {"Recursive": "tab:red", "Human_Control": "tab:blue"}

for cond in g0.index:
  y0, y3 = g0[cond] * 100, g3[cond] * 100
  ax.plot([0, 1], [y0, y3], marker="o", markersize=10,
          linewidth=3, color=colors[cond], label=cond)
  ax.annotate(f"{y0:.1f}%", (0, y0), xytext=(-10, 0),
              textcoords="offset points", ha="right", va="center", fontweight="bold")
  ax.annotate(f"{y3:.1f}%", (1, y3), xytext=(10, 0),
              textcoords="offset points", ha="left", va="center", fontweight="bold")

ax.set_xlim(-0.3, 1.3)
ax.set_xticks([0, 1])
ax.set_xticklabels(["G0\n(baseline)", "G3\n(after 3 generations)"])
ax.set_ylabel("Context Supported Rate (%)")
ax.set_title(
    "Crossover: Both Start ~48%, End 31 Points Apart\n"
    "(Context Supported Rate, G0 vs G3)",
    fontweight="bold",
)
ax.legend(loc="center left", bbox_to_anchor=(1.0, 0.5))
ax.spines[["top", "right"]].set_visible(False)

plt.tight_layout()
plt.savefig(PLOTS_DIR/"crossover_slope.png")
plt.close()

print("Saved crossover_slope.png")
print(f"Human_Control: {g0['Human_Control']*100:.1f}% -> {g3['Human_Control']*100:.1f}%  ({(g3['Human_Control']-g0['Human_Control'])*100:+.1f} pts)")
print(f"Recursive:     {g0['Recursive']*100:.1f}% -> {g3['Recursive']*100:.1f}%  ({(g3['Recursive']-g0['Recursive'])*100:+.1f} pts)")

## Combine cross condition cross seed micro bootstrap, micro summary, statistical analysis in one csv

In [ ]:
from pathlib import Path

import pandas as pd

RUN_PATHS = {
    ("Recursive", 42): Path(f"{RESULTS_DIR}/seeds/seed_42/recursive/evaluation"),
    ("Recursive", 123): Path(f"{RESULTS_DIR}/seeds/seed_123/recursive/evaluation"),
    ("Human_Control", 42): Path(f"{RESULTS_DIR}/seeds/seed_42/human_control/evaluation"),
    ("Human_Control", 123): Path(f"{RESULTS_DIR}/seeds/seed_123/human_control/evaluation"),
}

OUTPUT_DIR = DATA_DIR
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def combine_seed_file(file_pattern, output_name):

    records = []

    for (condition, seed), folder in RUN_PATHS.items():

        # Replace {seed} with 42 or 123

        filename = file_pattern.format(seed=seed)

        file = folder / filename

        if not file.exists():

            print(f"WARNING - File not found: {file}")

            continue

        df = pd.read_csv(file)

        # Add experiment identifiers

        df.insert(0, "condition", condition)

        df.insert(1, "seed", seed)

        records.append(df)

        print(

            f"Loaded: {condition}, seed {seed} "

            f"({len(df):,} rows)"

        )

    if not records:

        raise FileNotFoundError(

            f"No files found for pattern: {file_pattern}"

        )

    master = pd.concat(records, ignore_index=True)

    output_file = OUTPUT_DIR / output_name

    master.to_csv(output_file, index=False)

    print(f"\nCreated: {output_file}")

    print(f"Total rows: {len(master):,}")

    print(f"Columns: {len(master.columns)}")

    return master

In [ ]:
entity_summary = combine_seed_file(
    "seed{seed}_entity_micro_summary.csv",
    "cross_condition_cross_seed_entity_micro_summary_master.csv"
)

bootstrap = combine_seed_file(
    "seed{seed}_entity_micro_bootstrap_statistics.csv",
    "cross_condition_cross_seed_entity_micro_bootstrap_statistics_master.csv"
)

paired = combine_seed_file(
    "seed{seed}_paired_statistical_analysis.csv",
    "cross_condition_cross_seed_paired_statistical_analysis_master.csv"
)

## Create plots for cross_condition_cross_seed_entity_micro_bootstrap_statistics_master.csv

### 1. F1 across generations — main plot

X-axis: G0, G1, G2, G3
Y-axis: Micro F1
Lines: Recursive vs Human Control
Separate panels: Disease and Chemical

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

# 1. Load your CSV
CSV_PATH = DATA_DIR/"cross_condition_cross_seed_entity_micro_bootstrap_statistics_master.csv"

df = pd.read_csv(CSV_PATH)

df.columns = df.columns.str.replace("*", "", regex=False).str.strip()

# 2. Keep only the data needed for this plot

plot_df = df[
    df["comparison"].isin(["G0 → G1", "G1 → G2", "G2 → G3", "G0 → G3"])
].copy()

# 3. Create generation-level data
# The summary CSV gives us start/end F1 for each comparison.
# We reconstruct G0, G1, G2, G3 from those values.

records = []

for _, row in plot_df.iterrows():
    # Starting generation
    start_generation = row["comparison"].split(" → ")[0]
    end_generation = row["comparison"].split(" → ")[1]

    records.append({
        "condition": row["condition"],
        "seed": row["seed"],
        "entity_type": row["entity_type"],
        "generation": start_generation,
        "micro_f1": row["micro_f1_start"]
    })

    records.append({
        "condition": row["condition"],
        "seed": row["seed"],
        "entity_type": row["entity_type"],
        "generation": end_generation,
        "micro_f1": row["micro_f1_end"]
    })

generation_df = pd.DataFrame(records)

# Remove duplicate generation rows created by overlapping comparisons
generation_df = generation_df.drop_duplicates(
    subset=["condition", "seed", "entity_type", "generation"]
)

# 4. Average across seeds
mean_df = (
    generation_df
    .groupby(
        ["condition", "entity_type", "generation"],
        as_index=False
    )["micro_f1"]
    .mean()
)

# Put generations in correct order
generation_order = ["G0", "G1", "G2", "G3"]

mean_df["generation"] = pd.Categorical(
    mean_df["generation"],
    categories=generation_order,
    ordered=True
)

mean_df = mean_df.sort_values("generation")

# 5. Create Disease plot
# Colors
recursive_color = "#B52B27" # red
human_control_color = "#014C8C" # blue
plt.figure(figsize=(8, 6))

for condition in ["Recursive", "Human_Control", "Human Control"]:

    subset = mean_df[
        (mean_df["condition"] == condition) &
        (mean_df["entity_type"].str.lower() == "disease")
    ]

    if len(subset) == 0:
        continue

    # Display Human_Control nicely
    label = "Human Control" if condition in ["Human_Control", "Human Control"] else condition

    plt.plot(
        subset["generation"],
        subset["micro_f1"],
        marker="o",
        linewidth=2,
        markersize=7,
        color=recursive_color if condition == "Recursive" else human_control_color,
        label=label
    )

plt.xlabel("Generation", fontsize=12)
plt.ylabel("Micro F1", fontsize=12)
plt.title("Disease Entity Micro F1 Across Generations", fontsize=14)

plt.ylim(0, 0.30)

plt.grid(
    True,
    linestyle="--",
    alpha=0.3
)

plt.legend()
plt.tight_layout()

plt.savefig(
    PLOTS_DIR / "disease_micro_f1_across_generations.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()
plt.close()

# 7. Create Chemical plot
plt.figure(figsize=(8, 6))

for condition in ["Recursive", "Human_Control", "Human Control"]:

    subset = mean_df[
        (mean_df["condition"] == condition) &
        (mean_df["entity_type"].str.lower() == "chemical")
    ]

    if len(subset) == 0:
        continue

    label = "Human Control" if condition in ["Human_Control", "Human Control"] else condition

    plt.plot(
        subset["generation"],
        subset["micro_f1"],
        marker="o",
        linewidth=2,
        markersize=7,
        color=recursive_color if condition == "Recursive" else human_control_color,
        label=label
    )

plt.xlabel("Generation", fontsize=12)
plt.ylabel("Micro F1", fontsize=12)
plt.title("Chemical Entity Micro F1 Across Generations", fontsize=14)

plt.ylim(0, 0.30)

plt.grid(
    True,
    linestyle="--",
    alpha=0.3
)

plt.legend()
plt.tight_layout()

plt.savefig(
    PLOTS_DIR / "chemical_micro_f1_across_generations.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()
plt.close()

# 8. Save the averaged data
mean_df.to_csv(
    DATA_DIR / "micro_f1_generation_means.csv",
    index=False
)

print("Done!")
print(f"Plots saved to: {PLOTS_DIR.resolve()}")
print()
print(mean_df)

### 2. G0 → G3 overall change — very important

X-axis: Condition
Y-axis: Change in micro F1
Groups: Disease / Chemical
Points: Seed 42 and Seed 123

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

# 1. Load CSV

CSV_PATH = DATA_DIR / "cross_condition_cross_seed_entity_micro_bootstrap_statistics_master.csv"

df = pd.read_csv(CSV_PATH)

# Clean column names
df.columns = df.columns.str.replace("*", "", regex=False).str.strip()

# 2. Keep only G0 → G3 comparisons
plot_df = df[
    df["comparison"] == "G0 → G3"
].copy()

# 3. Clean condition names
plot_df["condition"] = plot_df["condition"].replace({
    "Human_Control": "Human Control"
})

# 4. Colors
# Recursive = shades of red
rec_colors = {
    42: "#D9534F",
    123: "#8C1B17"
}

# Human Control = shades of blue
ctrl_colors = {
    42: "#0275D8",
    123: "#01325C"
}

# 5. Create plot
fig, ax = plt.subplots(figsize=(9, 6))

conditions = ["Recursive", "Human Control"]
entity_types = ["disease", "chemical"]

# X positions
condition_positions = {
    "Recursive": 0,
    "Human Control": 1
}

# Small horizontal offsets so the two seeds don't overlap
seed_offsets = {
    42: -0.10,
    123: 0.10
}

# Plot each entity type and seed
for entity_type in entity_types:
    for seed in [42, 123]:
        subset = plot_df[
            (plot_df["condition"] == "Recursive") &
            (plot_df["entity_type"].str.lower() == entity_type) &
            (plot_df["seed"] == seed)
        ]

        if len(subset) > 0:
            x = condition_positions["Recursive"] + seed_offsets[seed]
            y = subset["mean_change"].iloc[0]

            ax.scatter(
                x,
                y,
                s=100,
                color=rec_colors[seed],
                edgecolor="black",
                linewidth=0.8,
                label=f"Recursive — Seed {seed}" if entity_type == "disease" else None,
                zorder=3
            )

            # Add entity type label next to point
            ax.annotate(
                entity_type.capitalize(),
                (x, y),
                xytext=(0, 9),
                textcoords="offset points",
                ha="center",
                fontsize=9
            )

        subset = plot_df[
            (plot_df["condition"] == "Human Control") &
            (plot_df["entity_type"].str.lower() == entity_type) &
            (plot_df["seed"] == seed)
        ]

        if len(subset) > 0:
            x = condition_positions["Human Control"] + seed_offsets[seed]
            y = subset["mean_change"].iloc[0]
            ax.scatter(
                x,
                y,
                s=100,
                color=ctrl_colors[seed],
                edgecolor="black",
                linewidth=0.8,
                label=f"Human Control — Seed {seed}" if entity_type == "disease" else None,
                zorder=3
            )

            ax.annotate(
                entity_type.capitalize(),
                (x, y),
                xytext=(0, 9),
                textcoords="offset points",
                ha="center",
                fontsize=9
            )

# 6. Zero reference line
ax.axhline(
    0,
    linestyle="--",
    linewidth=1.2,
    alpha=0.7
)

# 7. Axis formatting
ax.set_xticks([0, 1])
ax.set_xticklabels(
    ["Recursive", "Human Control"],
    fontsize=11
)

ax.set_xlabel(
    "Condition",
    fontsize=12
)

ax.set_ylabel(
    "Change in Micro F1 (G0 → G3)",
    fontsize=12
)

ax.set_title(
    "Overall Micro F1 Change Across Generations",
    fontsize=14
)

# Show change as percentage points
ax.yaxis.set_major_formatter(
    plt.FuncFormatter(lambda y, _: f"{y:.2f}")
)

ax.grid(
    axis="y",
    linestyle="--",
    alpha=0.3
)

# 8. Legend
ax.legend(
    loc="best",
    frameon=True
)

plt.tight_layout()

# 9. Save
output_path = PLOTS_DIR / "micro_f1_g0_g3_overall_change.png"

plt.savefig(
    output_path,
    dpi=300,
    bbox_inches="tight"
)

plt.show()
plt.close()

print(f"Saved: {output_path.resolve()}")

### 3. Generation-to-generation change

X-axis: transition

* G0 → G1
* G1 → G2
* G2 → G3

Y-axis: change in micro-F1

Lines/groups: Recursive vs Human Control
Panels: Disease / Chemical

Why it matters: first plot shows where F1 ends up, while this shows when the degradation happens.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

CSV_PATH = DATA_DIR / "cross_condition_cross_seed_entity_micro_bootstrap_statistics_master.csv"

df = pd.read_csv(CSV_PATH)

df.columns = df.columns.str.replace("*", "", regex=False).str.strip()

transitions = ["G0 → G1", "G1 → G2", "G2 → G3"]

plot_df = df[
    df["comparison"].isin(transitions)
].copy()

plot_df["condition"] = plot_df["condition"].replace({
    "Human_Control": "Human Control"
})

plot_df["comparison"] = pd.Categorical(
    plot_df["comparison"],
    categories=transitions,
    ordered=True
)

plot_df = plot_df.sort_values("comparison")

rec_colors = {
    42: "#D9534F",
    123: "#8C1B17"
}

ctrl_colors = {
    42: "#0275D8",
    123: "#01325C"
}

fig, axes = plt.subplots(
    1,
    2,
    figsize=(14, 6),
    sharey=True
)

entity_types = ["disease", "chemical"]

for ax, entity_type in zip(axes, entity_types):

    for condition in ["Recursive", "Human Control"]:

        for seed in [42, 123]:

            subset = plot_df[
                plot_df["condition"].isin([condition])
                & plot_df["seed"].isin([seed])
                & plot_df["entity_type"].str.lower().isin([entity_type])
            ].copy()

            if subset.empty:
                continue

            subset = subset.sort_values("comparison")

            if condition in ["Recursive"]:
                color = rec_colors[seed]
            else:
                color = ctrl_colors[seed]

            ax.plot(
                subset["comparison"],
                subset["mean_change"],
                marker="o",
                linewidth=2,
                markersize=7,
                color=color,
                label=f"{condition} — Seed {seed}"
            )

    ax.axhline(
        0,
        linestyle="--",
        linewidth=1.2,
        alpha=0.7
    )

    ax.set_title(
        entity_type.capitalize(),
        fontsize=14
    )

    ax.set_xlabel(
        "Generation Transition",
        fontsize=12
    )

    ax.grid(
        axis="y",
        linestyle="--",
        alpha=0.3
    )

axes[0].set_ylabel(
    "Change in Micro F1",
    fontsize=12
)

fig.suptitle(
    "Generation-to-Generation Change in Entity Micro F1",
    fontsize=16,
    y=1.02
)

handles, labels = axes[0].get_legend_handles_labels()

fig.legend(
    handles,
    labels,
    loc="upper center",
    bbox_to_anchor=(0.5, -0.01),
    ncol=2,
    frameon=True
)

plt.tight_layout()

output_path = PLOTS_DIR / "generation_to_generation_micro_f1_change.png"

plt.savefig(
    output_path,
    dpi=300,
    bbox_inches="tight"
)

plt.show()
plt.close()

print(f"Saved: {output_path.resolve()}")

### 4. Forest plot with 95% CI

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

CSV_PATH = DATA_DIR / "cross_condition_cross_seed_entity_micro_bootstrap_statistics_master.csv"

df = pd.read_csv(CSV_PATH)

df.columns = df.columns.str.replace("*", "", regex=False).str.strip()

plot_df = df[
    df["comparison"].isin(["G0 → G3"])
].copy()

plot_df["condition"] = plot_df["condition"].replace({
    "Human_Control": "Human Control"
})

plot_df["group"] = (
    plot_df["condition"]
    + " / "
    + plot_df["entity_type"].str.capitalize()
)

group_order = [
    "Recursive / Disease",
    "Recursive / Chemical",
    "Human Control / Disease",
    "Human Control / Chemical"
]

plot_df["group"] = pd.Categorical(
    plot_df["group"],
    categories=group_order,
    ordered=True
)

plot_df = plot_df.sort_values(["group", "seed"])

rec_colors = {
    42: "#D9534F",
    123: "#8C1B17"
}

ctrl_colors = {
    42: "#0275D8",
    123: "#01325C"
}

fig, ax = plt.subplots(
    figsize=(10, 7)
)

y_positions = range(len(group_order))

for y, group in zip(y_positions, group_order):

    subset = plot_df[
        plot_df["group"].isin([group])
    ].copy()

    if subset.empty:
        continue

    for _, row in subset.iterrows():

        if row["condition"] in ["Recursive"]:
            color = rec_colors[row["seed"]]
        else:
            color = ctrl_colors[row["seed"]]

        ax.errorbar(
            row["mean_change"],
            y,
            xerr=[[
                row["mean_change"] - row["ci_95_low"]
            ], [
                row["ci_95_high"] - row["mean_change"]
            ]],
            fmt="o",
            markersize=8,
            capsize=5,
            elinewidth=2,
            linewidth=1.5,
            color=color
        )

ax.axvline(
    0,
    linestyle="--",
    linewidth=1.2,
    alpha=0.7
)

ax.set_yticks(list(y_positions))
ax.set_yticklabels(group_order)

ax.set_xlabel(
    "Change in Micro F1 (G0 → G3)",
    fontsize=12
)

ax.set_ylabel(
    "Condition / Entity Type",
    fontsize=12
)

ax.set_title(
    "Overall Change in Entity Micro F1 with 95% CI",
    fontsize=15
)

ax.grid(
    axis="x",
    linestyle="--",
    alpha=0.3
)

plt.tight_layout()

output_path = (
    PLOTS_DIR /
    "forest_plot_micro_f1_g0_g3_95ci.png"
)

plt.savefig(
    output_path,
    dpi=300,
    bbox_inches="tight"
)

plt.show()
plt.close()

print(f"Saved: {output_path.resolve()}")

## Create plots for cross_condition_cross_seed_entity_micro_summary_master.csv

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

CSV_PATH = DATA_DIR / "cross_condition_cross_seed_entity_micro_summary_master.csv"

df = pd.read_csv(CSV_PATH)

df.columns = (
    df.columns
    .str.replace("*", "", regex=False)
    .str.strip()
)

df["condition"] = df["condition"].replace({
    "Human_Control": "Human Control"
})

df["generation"] = pd.Categorical(
    df["generation"],
    categories=["G0", "G1", "G2", "G3"],
    ordered=True
)

plot_df = (
    df.groupby(
        ["condition", "generation"],
        as_index=False
    )[
        [
            "disease_precision",
            "disease_recall",
            "disease_f1",
            "chemical_precision",
            "chemical_recall",
            "chemical_f1"
        ]
    ]
    .mean()
)

recursive_color = "#B52B27"
control_color = "#014C8C"

conditions = ["Recursive", "Human Control"]
generations = ["G0", "G1", "G2", "G3"]

fig, axes = plt.subplots(
    1,
    2,
    figsize=(14, 6),
    sharey=True
)

for ax, entity_type in zip(axes, ["disease", "chemical"]):
    for condition in conditions:
        subset = plot_df[
            plot_df["condition"].isin([condition])
        ].sort_values("generation")

        if entity_type in ["disease"]:
            y_values = subset["disease_f1"]
        else:
            y_values = subset["chemical_f1"]

        if condition in ["Recursive"]:
            color = recursive_color
        else:
            color = control_color

        ax.plot(
            subset["generation"],
            y_values,
            marker="o",
            linewidth=2.5,
            markersize=7,
            color=color,
            label=condition
        )

    ax.set_title(
        entity_type.capitalize(),
        fontsize=14
    )
    ax.set_xlabel(
        "Generation",
        fontsize=12
    )
    ax.grid(
        axis="y",
        linestyle="--",
        alpha=0.3
    )

axes[0].set_ylabel(
    "Micro F1",
    fontsize=12
)

fig.suptitle(
    "Entity Micro-F1 Across Generations",
    fontsize=16
)

handles, labels = axes[0].get_legend_handles_labels()

fig.legend(
    handles,
    labels,
    loc="upper center",
    bbox_to_anchor=(0.5, -0.01),
    ncol=2,
    frameon=True
)

plt.tight_layout()

output_path = PLOTS_DIR / "entity_micro_f1_across_generations.png"

plt.savefig(
    output_path,
    dpi=300,
    bbox_inches="tight"
)

plt.show()
plt.close()

print(f"Saved: {output_path.resolve()}")

fig, axes = plt.subplots(
    1,
    2,
    figsize=(14, 6),
    sharey=True
)

for ax, entity_type in zip(axes, ["disease", "chemical"]):
    for condition in conditions:
        subset = plot_df[
            plot_df["condition"].isin([condition])
        ].sort_values("generation")

        if entity_type in ["disease"]:
            precision_values = subset["disease_precision"]
            recall_values = subset["disease_recall"]
        else:
            precision_values = subset["chemical_precision"]
            recall_values = subset["chemical_recall"]

        if condition in ["Recursive"]:
            color = recursive_color
        else:
            color = control_color

        ax.plot(
            subset["generation"],
            precision_values,
            marker="o",
            linewidth=2,
            markersize=6,
            linestyle="-",
            color=color,
            label=f"{condition} — Precision"
        )

        ax.plot(
            subset["generation"],
            recall_values,
            marker="o",
            linewidth=2,
            markersize=6,
            linestyle="--",
            color=color,
            label=f"{condition} — Recall"
        )

    ax.set_title(
        entity_type.capitalize(),
        fontsize=14
    )
    ax.set_xlabel(
        "Generation",
        fontsize=12
    )
    ax.grid(
        axis="y",
        linestyle="--",
        alpha=0.3
    )

axes[0].set_ylabel(
    "Score",
    fontsize=12
)

fig.suptitle(
    "Entity Precision and Recall Across Generations",
    fontsize=16
)

handles, labels = axes[0].get_legend_handles_labels()

fig.legend(
    handles,
    labels,
    loc="upper center",
    bbox_to_anchor=(0.5, -0.01),
    ncol=2,
    frameon=True
)

plt.tight_layout()

output_path = PLOTS_DIR / "entity_precision_recall_across_generations.png"

plt.savefig(
    output_path,
    dpi=300,
    bbox_inches="tight"
)

plt.show()
plt.close()

print(f"Saved: {output_path.resolve()}")

## Create plots for cross_condition_cross_seed_paired_statistical_analysis.csv

### 1. ΔF1 with 95% CI — most important

* X-axis: Mean change in F1
* Y-axis: G0→G1, G1→G2, G2→G3, G0→G3
* Separate Disease F1 and Chemical F1 panels
* Recursive vs Human Control
* Horizontal 0 line
* Error bars = 95% CI
This shows how much F1 changes and the uncertainty around the change.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

INPUT_FILE = DATA_DIR / "cross_condition_cross_seed_paired_statistical_analysis_master.csv"

df = pd.read_csv(INPUT_FILE)

df = df[df["metric"].isin(["Disease F1", "Chemical F1"])].copy()

comparisons = ["G0 → G1", "G1 → G2", "G2 → G3", "G0 → G3"]
df = df[df["comparison"].isin(comparisons)].copy()

df["comparison"] = pd.Categorical(
    df["comparison"],
    categories=comparisons,
    ordered=True
)

condition_order = ["Recursive", "Human_Control"]

df["condition"] = pd.Categorical(
    df["condition"],
    categories=condition_order,
    ordered=True
)

recursive_color = "#B52B27"       # red
human_control_color = "#014C8C"   # blue

condition_colors = {
    "Recursive": recursive_color,
    "Human_Control": human_control_color
}


def plot_delta_f1(metric_name, output_name):
    plot_df = df[df["metric"] == metric_name].copy()

    fig, ax = plt.subplots(figsize=(10, 7))

    offsets = {
        "Recursive": 0.16,
        "Human_Control": -0.16
    }

    y_base = {
        "G0 → G1": 3,
        "G1 → G2": 2,
        "G2 → G3": 1,
        "G0 → G3": 0
    }

    for condition in condition_order:
        condition_df = plot_df[
            plot_df["condition"] == condition
        ].sort_values("comparison")

        x = []
        y = []
        xerr_low = []
        xerr_high = []

        for _, row in condition_df.iterrows():
            mean_change = row["mean_change"]
            ci_low = row["ci_95_low"]
            ci_high = row["ci_95_high"]

            x.append(mean_change)
            y.append(
                y_base[row["comparison"]] + offsets[condition]
            )

            xerr_low.append(mean_change - ci_low)
            xerr_high.append(ci_high - mean_change)

        ax.errorbar(
            x,
            y,
            xerr=[xerr_low, xerr_high],
            fmt="o",
            color=condition_colors[condition],
            ecolor=condition_colors[condition],
            capsize=4,
            markersize=7,
            linewidth=1.5,
            label=condition.replace("_", " ")
        )

    ax.axvline(
        x=0,
        color="black",
        linestyle="--",
        linewidth=1.2
    )

    ax.set_yticks(
        [3, 2, 1, 0],
        ["G0 → G1", "G1 → G2", "G2 → G3", "G0 → G3"]
    )

    ax.set_xlabel("Mean change in F1 (ΔF1)")
    ax.set_ylabel("Generation comparison")

    ax.set_title(
        f"{metric_name}: Change Across Generations with 95% CI"
    )

    ax.legend(
        title="Condition"
    )

    ax.grid(
        axis="x",
        alpha=0.25
    )

    plt.tight_layout()

    output_path = PLOTS_DIR / output_name

    plt.savefig(
        output_path,
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()

    print(f"Saved: {output_path}")


plot_delta_f1(
    "Disease F1",
    "delta_f1_disease_95ci.png"
)

plot_delta_f1(
    "Chemical F1",
    "delta_f1_chemical_95ci.png"
)

### 2. Relative F1 change (%)

* X-axis: Relative change %
* Y-axis: same four comparisons
* Disease and Chemical separately
* Recursive vs Human Control
* 0% reference line

This makes the magnitude easier to interpret. For example, Recursive seed 42 Disease F1 has −12.0% relative change from G0→G3.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

INPUT_FILE = DATA_DIR / "cross_condition_cross_seed_paired_statistical_analysis_master.csv"

df = pd.read_csv(INPUT_FILE)

df = df[df["metric"].isin(["Disease F1", "Chemical F1"])].copy()

comparisons = ["G0 → G1", "G1 → G2", "G2 → G3", "G0 → G3"]

df = df[df["comparison"].isin(comparisons)].copy()

df["comparison"] = pd.Categorical(
    df["comparison"],
    categories=comparisons,
    ordered=True
)

condition_order = ["Recursive", "Human_Control"]

df["condition"] = pd.Categorical(
    df["condition"],
    categories=condition_order,
    ordered=True
)

recursive_color = "#B52B27"       # red
human_control_color = "#014C8C"   # blue

condition_colors = {
    "Recursive": recursive_color,
    "Human_Control": human_control_color
}

def plot_relative_f1(metric_name, output_name):
    plot_df = df[df["metric"] == metric_name].copy()

    fig, ax = plt.subplots(figsize=(10, 7))

    offsets = {
        "Recursive": 0.16,
        "Human_Control": -0.16
    }

    y_base = {
        "G0 → G1": 3,
        "G1 → G2": 2,
        "G2 → G3": 1,
        "G0 → G3": 0
    }

    for condition in condition_order:
        condition_df = plot_df[
            plot_df["condition"] == condition
        ].sort_values("comparison")

        x = []
        y = []

        for _, row in condition_df.iterrows():
            x.append(row["relative_change_percent"])
            y.append(
                y_base[row["comparison"]] + offsets[condition]
            )

        ax.scatter(
            x,
            y,
            s=70,
            color=condition_colors[condition],
            label=condition.replace("_", " ")
        )

    ax.axvline(
        x=0,
        color="black",
        linestyle="--",
        linewidth=1.2
    )

    ax.set_yticks(
        [3, 2, 1, 0],
        ["G0 → G1", "G1 → G2", "G2 → G3", "G0 → G3"]
    )

    ax.set_xlabel("Relative change in F1 (%)")
    ax.set_ylabel("Generation comparison")

    ax.set_title(
        f"{metric_name}: Relative Change Across Generations"
    )

    ax.legend(
        title="Condition"
    )

    ax.grid(
        axis="x",
        alpha=0.25
    )

    plt.tight_layout()

    output_path = PLOTS_DIR / output_name

    plt.savefig(
        output_path,
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()

    print(f"Saved: {output_path}")


plot_relative_f1(
    "Disease F1",
    "relative_f1_change_disease.png"
)

plot_relative_f1(
    "Chemical F1",
    "relative_f1_change_chemical.png"
)

### 3. Cohen’s dz effect size

* X-axis: Cohen’s dz
* Y-axis: same comparisons
* Disease and Chemical separately
* Recursive vs Human Control
* 0 reference line

This gives you the standardized effect size alongside the raw F1 change.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np


INPUT_FILE = DATA_DIR / "cross_condition_cross_seed_paired_statistical_analysis_master.csv"

df = pd.read_csv(INPUT_FILE)

df = df[df["metric"].isin(["Disease F1", "Chemical F1"])].copy()

comparisons = ["G0 → G1", "G1 → G2", "G2 → G3", "G0 → G3"]
df = df[df["comparison"].isin(comparisons)].copy()

df["comparison"] = pd.Categorical(
    df["comparison"],
    categories=comparisons,
    ordered=True
)

condition_order = ["Recursive", "Human_Control"]

df["condition"] = pd.Categorical(
    df["condition"],
    categories=condition_order,
    ordered=True
)

recursive_color = "#B52B27"       # red
human_control_color = "#014C8C"   # blue

condition_colors = {
    "Recursive": recursive_color,
    "Human_Control": human_control_color
}


def plot_cohens_dz(metric_name, output_name):
    plot_df = df[df["metric"] == metric_name].copy()

    fig, ax = plt.subplots(figsize=(10, 7))

    offsets = {
        "Recursive": 0.16,
        "Human_Control": -0.16
    }

    y_base = {
        "G0 → G1": 3,
        "G1 → G2": 2,
        "G2 → G3": 1,
        "G0 → G3": 0
    }

    for condition in condition_order:
        condition_df = plot_df[
            plot_df["condition"] == condition
        ].sort_values("comparison")

        x = []
        y = []

        for _, row in condition_df.iterrows():
            x.append(row["cohens_dz"])
            y.append(
                y_base[row["comparison"]] + offsets[condition]
            )

        ax.scatter(
            x,
            y,
            s=70,
            color=condition_colors[condition],
            label=condition.replace("_", " ")
        )

    ax.axvline(
        x=0,
        color="black",
        linestyle="--",
        linewidth=1.2
    )

    ax.set_yticks(
        [3, 2, 1, 0],
        ["G0 → G1", "G1 → G2", "G2 → G3", "G0 → G3"]
    )

    ax.set_xlabel("Cohen's dz")
    ax.set_ylabel("Generation comparison")

    ax.set_title(
        f"{metric_name}: Cohen's dz Effect Size"
    )

    ax.legend(
        title="Condition"
    )

    ax.grid(
        axis="x",
        alpha=0.25
    )

    plt.tight_layout()

    output_path = PLOTS_DIR / output_name

    plt.savefig(
        output_path,
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()

    print(f"Saved: {output_path}")


plot_cohens_dz(
    "Disease F1",
    "cohens_dz_disease.png"
)

plot_cohens_dz(
    "Chemical F1",
    "cohens_dz_chemical.png"
)